# Amortized policy improvement, round 2: the 0006 value decoder (Phase A)

## Description

<!-- Written 2026-11-11 before the first run (pre-registration). Design: research_log/decisions/0006 §3, §4 Phase A; round 1: notebook 06. Issue #13. -->

**Question.** Round 1 (notebook 06, NO-GO) had clairvoyant headroom over EI: 0.70 % of the gap per state, carried by the top 10% of states. But its value head could not rank candidates (validation Spearman −0.01), and its conservative rule fell back to EI.

Does the 0006 value decoder learn a usable advantage from the same kind of labels, and does one step of policy improvement with it beat EI?

**Setup.**
- *Base policy, candidates, labels*: as round 1.
  - π₀ = EI on the frozen noise-free `pfn_large`; the candidate set C(s) has 133 candidates and x₀ = EI's pick.
  - Labels:
    - the 4000 pinned round-1 states (split 4, instances 0–3999), plus 8000 new states made the same way (instances 4000–11999);
    - 16 labeled candidates per state, 4 ground-truth continuations each, CRN across candidates;
    - d = 2, T = 20.
- *Decoder* (0006 §3):
  - candidate queries are a linear embedding of set-independent features of x (x; the PFN's marginal at x relative to the incumbent; EI, PI; distances; source flag);
  - no candidate–candidate attention;
  - 4 blocks of width 256. Block b cross-attends to the PFN's residual context states at layer 2b (the decoder's own LayerNorm and projection), plus a state token (budget), a statistics token (round 1's global features) and an x₀ token;
  - adaptive LayerNorm on the budget;
  - output: a 63-bin bar distribution (signed-log borders, see the revision below) over the single-seed paired advantage R(x₀, k) − R(x, k), in % of the gap, trained by NLL on every seed (not the seed mean). There is a bin of half-width 0.001 at 0.
- *Null*: the same decoder trained on targets permuted across each state's non-base candidates.
- *Rule* (0006 §3): deviate to the candidate with the largest score if the score exceeds 0.001 (the zero bin's upper edge), else EI's pick.
  - The score is a quantile of the predicted advantage (levels 0.1, 0.3, 0.5) or its mean.
  - rule\* = the best mean normalized regret on split 0 (300 tasks; ties go to the more conservative rule).
- *Test*: split 1, the same 1000 tasks as round 1 (same seeds, offset 0, CRN). Arms: EI, the decoder at every rule, the null decoder at rule\*.

**Deviation from 0006 §2.**
- Candidates are not yet refined by gradient ascent through the query path. That would change π₀, so the round-1 labels would no longer apply.
- x₀ is still exactly the rollouts' EI pick (`Engine.propose("ei")`, tested).

**Revision after the sanity run (2026-11-11 16:43, before the pilot).** Most single-seed advantages lie within ±0.002, and quantile borders collapsed to 25 bins.

The pilot uses signed-log borders instead:
- the zero bin [−0.001, 0.001];
- 31 log-spaced bins per side, up to 1.5 × the largest |a|; 63 bins in all.

On the round-1 labels, 59% of non-base single-seed targets fall in the zero bin, and the other bins hold up to 1.3% each. The rule and the decision criterion are unchanged.

The sanity also showed that the "mean" rule deviates almost always (99% of steps; heavy tails make the mean positive), with a test regret of 9.2 against EI's 1.0. The calibration on split 0 is what guards against this; it stays as pre-registered.

**What to look for.**
- On validation, the within-state Spearman between the predicted mean and the label (round 1: −0.01).
- How often the decoder deviates, from which sources, and in which steps.
- π₁ at rule\* against EI, paired by task.
- The same comparison against round 1's π₁ at c\* = 1, which ran on the same 1000 test tasks. Its EI arm must be identical, which the notebook checks.

**GO / NO-GO** (rule `analysis/api.py::api_verdict`, as round 1, on `paired_delta(test, "dec_<rule*>", "ei")` and the null):
- **INVALID**: the null beats EI with its CI below 0 (a gain without information means a bug or leakage).
- **GO**: the mean normalized regret at rule\* is lower by ≥ 5% relative, with the CI of the difference below 0.
- **NO-GO**: the CI rules out a 5% relative gain.
- **INCONCLUSIVE**: otherwise.

A GO starts Phase B planning (noisy two-head PFN) and gradient-refined candidates. A NO-GO with a clearly better ranking than round 1 points to more labels or the readout. A NO-GO without it points to the label noise (the split-half reliability of notebook 08) or to C(s).

**Limits.**
- One PFN, d = 2, T = 20, noise-free; the expected headroom is small here (0006 §4).
- Labels come from 4 continuations; the single-seed targets are heavy-tailed.
- 12 000 states for a model of about 3M parameters.
- One seed of decoder training.


In [ ]:
from anytime_acq.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/api_decoder/<sweep_id>"  # set after `just snapshot runs/api_decoder/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.